In [4]:
import pandas as pd
import numpy as np
from datetime import datetime


In [2]:
df = pd.read_csv('train.csv')

In [3]:
df.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [4]:
print(f"Shape: {df.shape}")


Shape: (3000888, 6)


In [5]:
print(f"\nFirst 5 rows:")
df.head()



First 5 rows:


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [6]:
df.dtypes

id               int64
date            object
store_nbr        int64
family          object
sales          float64
onpromotion      int64
dtype: object

In [7]:
df['date'] = pd.to_datetime(df['date'])

In [8]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])


In [9]:
print(df['date'].dtype)   # should show datetime64[ns]
print(df['date'].min(), df['date'].max())


datetime64[ns]
2013-01-01 00:00:00 2017-08-15 00:00:00


In [10]:
print(f"\nUnique stores: {df['store_nbr'].nunique()}")
print(f"Unique families: {df['family'].nunique()}")


Unique stores: 54
Unique families: 33


In [11]:
# CELL 6: Clean the data

# 1. Convert date to proper datetime
df['date'] = pd.to_datetime(df['date'])

# 2. Check for duplicates
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

# 3. Check for negative sales (shouldn't happen, but let's verify)
negative_sales = (df['sales'] < 0).sum()
print(f"Negative sales: {negative_sales}")

# 4. Look at sales distribution
print(f"\nSales statistics:")
df['sales'].describe()

# .describe() gives quick statistics: count, mean, std, min, 25%, 50%, 75%, max

Duplicate rows: 0
Negative sales: 0

Sales statistics:


count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
max      1.247170e+05
Name: sales, dtype: float64

In [12]:
# CELL 7: Handle missing data (practice, even if none exist)

# Strategy: For time series, we can't just drop missing dates.
# If January 5th is missing, we need to fill it in or our date sequence is broken.

# First, let's create a complete date range for one store-family combination
sample = df[(df['store_nbr'] == 1) & (df['family'] == 'GROCERY I')].copy()

# Create a complete date range
date_range = pd.date_range(start=df['date'].min(), end=df['date'].max(), freq='D')
# freq='D' means daily frequency

# Set date as index for reindexing
sample = sample.set_index('date')

# Reindex to fill missing dates with NaN
sample_complete = sample.reindex(date_range)

print(f"Original rows: {len(sample)}")
print(f"After filling missing dates: {len(sample_complete)}")
print(f"Missing values created: {sample_complete['sales'].isnull().sum()}")

# Fill missing sales with 0 (assuming no sales on missing days)
# In real life, you'd investigate why data is missing first!
sample_complete['sales'] = sample_complete['sales'].fillna(0)

# Forward fill other columns (carry last known value forward)
sample_complete['store_nbr'] = sample_complete['store_nbr'].fillna(method='ffill')
sample_complete['family'] = sample_complete['family'].fillna(method='ffill')
sample_complete['onpromotion'] = sample_complete['onpromotion'].fillna(0)

Original rows: 1684
After filling missing dates: 1688
Missing values created: 4


/tmp/ipykernel_70/3992385895.py:28: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  sample_complete['store_nbr'] = sample_complete['store_nbr'].fillna(method='ffill')
/tmp/ipykernel_70/3992385895.py:29: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  sample_complete['family'] = sample_complete['family'].fillna(method='ffill')


In [13]:
# CELL 8: Transform data to match our schema

# 1. Create products table data
# Each unique 'family' becomes a product in our simplified model
products_df = pd.DataFrame({
    'product_name': df['family'].unique(),
    'category': df['family'].unique(),  # In this dataset, family IS the category
    'unit_cost': 0.0  # We don't have cost data, so we use 0 as placeholder
})

print(f"Products to insert: {len(products_df)}")
products_df.head()

Products to insert: 33


,product_name,category,unit_cost
0,AUTOMOTIVE,AUTOMOTIVE,0.0
1,BABY CARE,BABY CARE,0.0
2,BEAUTY,BEAUTY,0.0
3,BEVERAGES,BEVERAGES,0.0
4,BOOKS,BOOKS,0.0


In [14]:
# CELL 9: Create stores table data
stores_df = pd.DataFrame({
    'store_name': [f"Store_{n}" for n in df['store_nbr'].unique()],
    'city': 'Unknown',  # Dataset doesn't include city
    'country': 'Ecuador',  # We know this from the dataset description
    'store_type': 'supermarket'
})

print(f"Stores to insert: {len(stores_df)}")
stores_df.head()

Stores to insert: 54


,store_name,city,country,store_type
0,Store_1,Unknown,Ecuador,supermarket
1,Store_10,Unknown,Ecuador,supermarket
2,Store_11,Unknown,Ecuador,supermarket
3,Store_12,Unknown,Ecuador,supermarket
4,Store_13,Unknown,Ecuador,supermarket


In [15]:
# CELL 10: Create sales_transactions data

# We need to map family names to product_ids and store numbers to store_ids
# Since we haven't inserted into the database yet, we'll create the mappings manually

# Create mapping dictionaries
family_to_id = {family: idx + 1 for idx, family in enumerate(df['family'].unique())}
store_nbr_to_id = {nbr: idx + 1 for idx, nbr in enumerate(df['store_nbr'].unique())}

sales_df = df.copy()
sales_df['product_id'] = sales_df['family'].map(family_to_id)
sales_df['store_id'] = sales_df['store_nbr'].map(store_nbr_to_id)

# Select and rename columns to match our schema
sales_df = sales_df[['product_id', 'store_id', 'date', 'sales', 'onpromotion']].copy()
sales_df.columns = ['product_id', 'store_id', 'transaction_date', 'units_sold', 'unit_price']

# unit_price doesn't exist in original data, we'll calculate a proxy
# (In real life, you'd have actual price data)
sales_df['unit_price'] = 1.0  # Placeholder

print(f"Sales transactions to insert: {len(sales_df)}")
sales_df.head()

Sales transactions to insert: 3000888


,product_id,store_id,transaction_date,units_sold,unit_price
0,1,1,2013-01-01,0.0,1.0
1,2,1,2013-01-01,0.0,1.0
2,3,1,2013-01-01,0.0,1.0
3,4,1,2013-01-01,0.0,1.0
4,5,1,2013-01-01,0.0,1.0


In [2]:
# CELL 11: Load data into database

from sqlalchemy import text
from database import get_engine

# Get our database engine
engine = get_engine()

# Load products
products_df.to_sql('products', engine, if_exists='append', index=False)
# to_sql: "Take this DataFrame and put it into a SQL table"
# if_exists='append': "Add to existing data" (the table is empty now, but good habit)
# index=False: "Don't include the DataFrame row numbers as a column"

print("✅ Products loaded")

# Load stores
stores_df.to_sql('stores', engine, if_exists='append', index=False)
print("✅ Stores loaded")

# Load sales (this might take 30-60 seconds for 3 million rows)
print("Loading sales... (this takes a while)")
sales_df.to_sql('sales_transactions', engine, if_exists='append', index=False)
print("✅ Sales loaded")

NameError: name 'products_df' is not defined

In [3]:
products_df = pd.read_csv("products.csv")
stores_df   = pd.read_csv("stores.csv")
sales_df    = pd.read_csv("sales.csv")


NameError: name 'pd' is not defined

In [5]:
products_df = pd.read_csv("products.csv")
stores_df   = pd.read_csv("stores.csv")
sales_df    = pd.read_csv("sales.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'products.csv'

In [6]:
print(products_df.shape)
print(stores_df.shape)
print(sales_df.shape)


NameError: name 'products_df' is not defined

In [7]:
!ls /app


Dockerfile	__pycache__  docker-compose.yml  schema.sql
Untitled.ipynb	database.py  requirements.txt	 train.csv


In [8]:
import pandas as pd
train_df = pd.read_csv("train.csv")
train_df.head()
train_df.columns


Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion'], dtype='object')

In [ ]:
sales_df = pd.read_csv("train.csv")
sales_df.to_sql('sales_transactions', engine, if_exists='append', index=False)


In [1]:
sales_df = pd.read_csv("train.csv")
sales_df.to_sql('sales_transactions', engine, if_exists='append', index=False)


NameError: name 'pd' is not defined